# Partie 1 : Import des packages 

In [134]:
import pandas as pd
import dash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import dash_bootstrap_components as dbc


# Partie 2 : Préparations des données

In [135]:
# Importer les données
df = pd.read_csv("data.csv")

# Colonnes d'intérêt
df = df[
[
"CustomerID",
"Gender",
"Location",
"Product_Category",
"Quantity",
"Avg_Price",
"Transaction_Date",
"Month",
"Discount_pct"
]
]

# Nettoyage des données, conversion des types et calcul du prix total
df["CustomerID"] = df["CustomerID"].fillna(0).astype(int)
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Avg_Price"] = pd.to_numeric(df["Avg_Price"], errors="coerce")
df["Discount_pct"] = pd.to_numeric(df["Discount_pct"], errors="coerce")

# Convertir la colonne "Transaction_Date" en datetime, en gérant les erreurs
df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"], errors="coerce")
df = df.dropna(subset=["Transaction_Date","Quantity","Avg_Price"])

# Calculer le prix total après remise
df["Total_price"] = df["Quantity"] * df["Avg_Price"] * (1 - df["Discount_pct"]/100)


### Partie 2 bis : verifications

In [136]:
df.head()
df.info()
df["Total_price"].head()

<class 'pandas.core.frame.DataFrame'>
Index: 52924 entries, 0 to 52923
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   CustomerID        52924 non-null  int64         
 1   Gender            52924 non-null  object        
 2   Location          52924 non-null  object        
 3   Product_Category  52924 non-null  object        
 4   Quantity          52924 non-null  float64       
 5   Avg_Price         52924 non-null  float64       
 6   Transaction_Date  52924 non-null  datetime64[ns]
 7   Month             52924 non-null  int64         
 8   Discount_pct      52524 non-null  float64       
 9   Total_price       52524 non-null  float64       
dtypes: datetime64[ns](1), float64(4), int64(2), object(3)
memory usage: 4.4+ MB


0    138.339
1    138.339
2    220.986
3     73.350
4    138.339
Name: Total_price, dtype: float64

# Partie 3 : Fonctions

In [137]:
# 1. Calcul du chiffre d'affaire
def calculer_chiffre_affaire(data):
    return data["Total_price"].sum()


# 2. Fréquence des meilleures ventes
def frequence_meilleure_vente(data, top=10, ascending=False):
    return data["Product_Category"].value_counts(ascending=ascending).head(top)


# 3. Indicateur du mois
def indicateur_du_mois(data, current_month=12):

    prev_month = 12 if current_month == 1 else current_month - 1

    current = data[data["Transaction_Date"].dt.month == current_month]
    previous = data[data["Transaction_Date"].dt.month == prev_month]

    return {
        "ca_current": calculer_chiffre_affaire(current),
        "ca_previous": calculer_chiffre_affaire(previous),
        "sales_current": len(current),
        "sales_previous": len(previous)
    }


### Partie 3 bis : Affichages / Verifications des fonctions 

In [138]:
print("1. Chiffre d'affaire total :")
print(calculer_chiffre_affaire(df))

print("\n2. Top 10 des ventes par catégorie :")
print(frequence_meilleure_vente(df))

print("\n3. Indicateur du mois :")
print(indicateur_du_mois(df))

1. Chiffre d'affaire total :
3717667.2469999995

2. Top 10 des ventes par catégorie :
Product_Category
Apparel                 18126
Nest-USA                14013
Office                   6513
Drinkware                3483
Lifestyle                3092
Nest                     2198
Bags                     1882
Headgear                  771
Notebooks & Journals      749
Waze                      554
Name: count, dtype: int64

3. Indicateur du mois :
{'ca_current': np.float64(366280.733), 'ca_previous': np.float64(406866.128), 'sales_current': 4502, 'sales_previous': 3961}


# Partie 4 : Initalisation de l'app dash 

In [139]:
# Initialiser l'application Dash, en utilisant le thème Bootstrap pour une meilleure apparence
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Partie 5 : Layout (interface)

In [140]:
# ici on s'est aidé de chatgpt pour faire un visuel épuré 
app.layout = html.Div([

    # HEADER
    html.Div([

        html.Div([
            html.H2("ECAP Store Dashboard", style={"color": "white"}),
            html.P("Analyse des ventes", style={"color": "white"})
        ]),

        dcc.Dropdown(
            id="location_filter",
            options=[{"label": i, "value": i} for i in sorted(df["Location"].unique())],
            multi=True,
            placeholder="Filtrer par zone",
            style={"width": "250px"}
        )

    ], style={
        "backgroundColor": "#4e73df",
        "padding": "15px",
        "display": "flex",
        "justifyContent": "space-between"
    }),

    # BODY
    html.Div([

        # -------- LEFT --------
        html.Div([

            # KPI
            html.Div([

                html.Div([
                    html.H4("Décembre"),
                    html.H1(id="kpi_ca"),
                    html.Div(id="kpi_ca_var")
                ], style={"width": "45%", "textAlign": "center"}),

                html.Div([
                    html.H4("Décembre"),
                    html.H1(id="kpi_sales"),
                    html.Div(id="kpi_sales_var")
                ], style={"width": "45%", "textAlign": "center"})

            ], style={"display": "flex", "justifyContent": "space-around"}),

            # BAR CHART
            dcc.Graph(id="bar_chart")

        ], style={"width": "45%", "padding": "10px"}),

        # -------- RIGHT --------
        html.Div([

            # LINE CHART
            dcc.Graph(id="line_chart"),

            # TABLE
            dash_table.DataTable(
                id="sales_table",
                page_size=10,
                filter_action="native",
                sort_action="native"
            )

        ], style={"width": "55%", "padding": "10px"})

    ], style={"display": "flex"})

])

# Partie 6 : Callback 

In [141]:
@app.callback(
    [
        Output("kpi_ca", "children"),
        Output("kpi_ca_var", "children"),
        Output("kpi_sales", "children"),
        Output("kpi_sales_var", "children"),
        Output("line_chart", "figure"),
        Output("bar_chart", "figure"),
        Output("sales_table", "data"),
        Output("sales_table", "columns")
    ],
    Input("location_filter", "value")
)
def update_dashboard(location):

    # FILTRAGE
    dff = df.copy()
    if location:
        dff = dff[dff["Location"].isin(location)]

  
    # KPI (avec fonction)
    indicateurs = indicateur_du_mois(dff)

    ca_dec = indicateurs["ca_current"]
    ca_nov = indicateurs["ca_previous"]

    sales_dec = indicateurs["sales_current"]
    sales_nov = indicateurs["sales_previous"]

    var_ca = ca_dec - ca_nov
    var_sales = sales_dec - sales_nov

    # affichage KPI
    kpi_ca = f"{ca_dec/1000:.0f}k"
    kpi_sales = f"{sales_dec}"

    kpi_ca_var = html.Span(
        f"▲ +{var_ca/1000:.0f}k" if var_ca >= 0 else f"▼ -{abs(var_ca)/1000:.0f}k",
        style={"color": "green" if var_ca >= 0 else "red"}
    )

    kpi_sales_var = html.Span(
        f"▲ +{var_sales}" if var_sales >= 0 else f"▼ -{abs(var_sales)}",
        style={"color": "green" if var_sales >= 0 else "red"}
    )

    # =========================
    # LINE CHART
    # =========================
    weekly = (
        dff.groupby(pd.Grouper(key="Transaction_Date", freq="W"))["Total_price"]
        .sum()
        .reset_index()
    )

    line = px.line(
        weekly,
        x="Transaction_Date",
        y="Total_price",
        title="Evolution du chiffre d'affaire par semaine",
        labels={
            "Transaction_Date": "Semaine",
            "Total_price": "Chiffre d'affaires"
        }
    )

    # =========================
    # BAR CHART
    # =========================
    dff_dec = dff[dff["Transaction_Date"].dt.month == 12]

    top_categories = frequence_meilleure_vente(dff_dec)

    dff_dec = dff_dec[dff_dec["Product_Category"].isin(top_categories.index)]

    df_grouped = (
        dff_dec.groupby(["Product_Category", "Gender"])
        .size()
        .unstack(fill_value=0)
    )

    # tri
    df_grouped["total"] = df_grouped.sum(axis=1)
    df_grouped = df_grouped.sort_values("total", ascending=False).head(10)
    df_grouped = df_grouped.drop(columns="total")

    df_plot = df_grouped.reset_index().melt(id_vars="Product_Category")

    bar = px.bar(
        df_plot,
        x="value",
        y="Product_Category",
        color="Gender",
        orientation="h",
        barmode="group",
        title="Top 10 des catégories - Nombre de commandes (Décembre)",
        labels={
            "value": "Nombre de commandes",
            "Product_Category": "Catégorie",
            "Gender": "Genre"
        },
        color_discrete_map={"F": "#FFB6C1", "M": "#ADD8E6"},
        category_orders={
            "Product_Category": df_grouped.index.tolist()
        }
    )

    # =========================
    # TABLE
    # =========================
    table = dff.sort_values("Transaction_Date", ascending=False).head(100).copy()

    table["Transaction_Date"] = table["Transaction_Date"].dt.strftime("%d/%m/%Y")
    table["Avg_Price"] = table["Avg_Price"].round(2)
    table["Total_price"] = table["Total_price"].round(2)

    columns = [
        {"name": "Date", "id": "Transaction_Date"},
        {"name": "Sexe", "id": "Gender"},
        {"name": "Ville", "id": "Location"},
        {"name": "Catégorie produit", "id": "Product_Category"},
        {"name": "Quantité", "id": "Quantity"},
        {"name": "Prix moyen (€)", "id": "Avg_Price"},
        {"name": "Remise (%)", "id": "Discount_pct"},
        {"name": "Prix total (€)", "id": "Total_price"}
    ]


    # RETURN

    return (
        kpi_ca,
        kpi_ca_var,
        kpi_sales,
        kpi_sales_var,
        line,
        bar,
        table.to_dict("records"),
        columns
    )



# Partie 7 : Lancement 

In [142]:
if __name__ == "__main__":
    app.run(debug=True, port=8056)

In [143]:
# =========================
# IMPORT
# =========================
import pandas as pd
import dash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px

# =========================
# DATA
# =========================
df = pd.read_csv("data.csv")

df = df[
[
"CustomerID",
"Gender",
"Location",
"Product_Category",
"Quantity",
"Avg_Price",
"Transaction_Date",
"Month",
"Discount_pct"
]
]

df["CustomerID"] = df["CustomerID"].fillna(0).astype(int)
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Avg_Price"] = pd.to_numeric(df["Avg_Price"], errors="coerce")
df["Discount_pct"] = pd.to_numeric(df["Discount_pct"], errors="coerce")

df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"], errors="coerce")
df = df.dropna(subset=["Transaction_Date","Quantity","Avg_Price"])

df["Total_price"] = df["Quantity"] * df["Avg_Price"] * (1 - df["Discount_pct"]/100)


# =========================
# APP
# =========================
app = dash.Dash(__name__)

# =========================
# LAYOUT
# =========================
app.layout = html.Div([

    # HEADER
    html.Div([
        html.Div([
            html.H2("ECAP Store Dashboard", style={"color":"white"}),
            html.P("Analyse des ventes", style={"color":"white"})
        ]),

        dcc.Dropdown(
            id="location_filter",
            options=[{"label":i,"value":i} for i in sorted(df["Location"].unique())],
            multi=True,
            placeholder="Filtrer par zone",
            style={"width":"250px"}
        )

    ], style={
        "backgroundColor":"#4e73df",
        "padding":"15px",
        "display":"flex",
        "justifyContent":"space-between"
    }),

    # BODY
    html.Div([

        # LEFT
        html.Div([

            html.Div([

                html.Div([
                    html.H4("Décembre"),
                    html.H1(id="kpi_ca"),
                    html.Div(id="kpi_ca_var")
                ], style={"width":"45%", "textAlign":"center"}),

                html.Div([
                    html.H4("Décembre"),
                    html.H1(id="kpi_sales"),
                    html.Div(id="kpi_sales_var")
                ], style={"width":"45%", "textAlign":"center"})

            ], style={"display":"flex", "justifyContent":"space-around"}),

            dcc.Graph(id="bar_chart")

        ], style={"width":"45%", "padding":"10px"}),

        # RIGHT
        html.Div([

            dcc.Graph(id="line_chart"),

            dash_table.DataTable(
                id="sales_table",
                page_size=10,
                filter_action="native",
                sort_action="native"
            )

        ], style={"width":"55%", "padding":"10px"})

    ], style={"display":"flex"})

])


# =========================
# CALLBACK
# =========================
@app.callback(
    [
        Output("kpi_ca","children"),
        Output("kpi_ca_var","children"),
        Output("kpi_sales","children"),
        Output("kpi_sales_var","children"),
        Output("line_chart","figure"),
        Output("bar_chart","figure"),
        Output("sales_table","data"),
        Output("sales_table","columns")
    ],
    Input("location_filter","value")
)

def update_dashboard(location):

    dff = df.copy()

    if location:
        dff = dff[dff["Location"].isin(location)]

    # KPI
    dec = dff[dff["Transaction_Date"].dt.month == 12]
    nov = dff[dff["Transaction_Date"].dt.month == 11]

    ca_dec = dec["Total_price"].sum()
    ca_nov = nov["Total_price"].sum()

    sales_dec = len(dec)
    sales_nov = len(nov)

    var_ca = ca_dec - ca_nov
    var_sales = sales_dec - sales_nov

    kpi_ca = f"{ca_dec/1000:.0f}k"
    kpi_sales = f"{sales_dec}"

    kpi_ca_var = html.Span(
        f"▲ +{var_ca/1000:.0f}k" if var_ca >= 0 else f"▼ -{abs(var_ca)/1000:.0f}k",
        style={"color":"green" if var_ca >= 0 else "red"}
    )

    kpi_sales_var = html.Span(
        f"▲ +{var_sales}" if var_sales >= 0 else f"▼ -{abs(var_sales)}",
        style={"color":"green" if var_sales >= 0 else "red"}
    )

    # LINE CHART
    weekly = dff.groupby(pd.Grouper(key="Transaction_Date", freq="W"))["Total_price"].sum().reset_index()

    line = px.line(
        weekly,
        x="Transaction_Date",
        y="Total_price",
        title="Évolution du chiffre d'affaires par semaine",
        labels={"Transaction_Date":"Semaine","Total_price":"Chiffre d'affaires"}
    )

   
    # BAR CHART

    # filtrer décembre
    dff_dec = dff[dff["Transaction_Date"].dt.month == 12]

    # grouper par catégorie + genre (nombre de commandes)
    df_grouped = (
        dff_dec.groupby(["Product_Category", "Gender"])
        .size()
        .unstack(fill_value=0)
    )

    # trier les 10 meilleures catégories
    df_grouped["total"] = df_grouped.sum(axis=1)
    df_grouped = df_grouped.sort_values("total", ascending=False).head(10)
    df_grouped = df_grouped.drop(columns="total")

    # transformer en format "long" pour plotly
    df_plot = df_grouped.reset_index().melt(id_vars="Product_Category")

    # graphique
    bar = px.bar(
        df_plot,
        x="value",
        y="Product_Category",
        color="Gender",
        orientation="h",
        barmode="group",

        title="Top 10 des catégories - Nombre de commandes (Décembre)",

        labels={
            "value": "Nombre de commandes",
            "Product_Category": "Catégorie",
            "Gender": "Genre"
        },

        color_discrete_map={
            "F": "#FFB6C1",
            "M": "#ADD8E6"
        },

        category_orders={
            "Product_Category": df_grouped.index.tolist()
        }
    )

    # TABLE
    table = dff.sort_values("Transaction_Date", ascending=False).head(100).copy()

    table["Transaction_Date"] = table["Transaction_Date"].dt.strftime("%d/%m/%Y")

    table["Avg_Price"] = table["Avg_Price"].round(2)
    table["Total_price"] = table["Total_price"].round(2)

    columns = [
        {"name":"Date","id":"Transaction_Date"},
        {"name":"Sexe","id":"Gender"},
        {"name":"Ville","id":"Location"},
        {"name":"Catégorie produit","id":"Product_Category"},
        {"name":"Quantité","id":"Quantity"},
        {"name":"Prix moyen (€)","id":"Avg_Price"},
        {"name":"Remise (%)","id":"Discount_pct"},
        {"name":"Prix total (€)","id":"Total_price"}
    ]

    return kpi_ca, kpi_ca_var, kpi_sales, kpi_sales_var, line, bar, table.to_dict("records"), columns


# =========================
# RUN
# =========================
if __name__ == "__main__":
    app.run(debug=True, port=8033)